In [10]:
# AVM Feature Engineering
# imports + settings. nothing happens to the data here, just paths, the random seed
# and the knobs (number of map zones, how far back the area lookup goes, which training window sizes to try, etc)
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100, "display.width", 200, "display.float_format", "{:,.3f}".format)

SEED = 42
DATA_DIR = Path(".")
OUT_DIR = Path("fe_outputs"); OUT_DIR.mkdir(exist_ok=True)
ALL_PATH = DATA_DIR / "data/sold/cleaned_all.csv"
MODEL_PATH = DATA_DIR / "data/sold/cleaned_model_input.csv"
SCHOOL_DISTRICT_PATH = DATA_DIR / "data/DistrictAreas2425/DistrictAreas2425.shp"

MISSING_FLAG_THRESHOLD = 0.005
N_GEO_CLUSTERS = 80
AREA_WINDOW, AREA_LAG, AREA_SHORT, AREA_MIN_N = 12, 1, 3, 5
WARMUP_MONTHS = 1
TRAIN_WINDOWS = [6, 9, 12, 15, 18, 21, 24, 27, 30]  # months of training data to try (val + test are extra, on top of these)
np.random.seed(SEED)

In [11]:
# 1. load
# read in the data (already cleaned in an earlier step) and sort the columns into
# target / categorical / numeric. work = the copy we build all the features on
full = pd.read_csv(ALL_PATH, low_memory=False, parse_dates=["CloseDate"])
model_in = pd.read_csv(MODEL_PATH, low_memory=False)
assert len(full) == len(model_in)

TARGET = "ClosePrice"
CAT_COLS = ["City", "CountyOrParish", "HighSchoolDistrict", "MLSAreaMajor"]
BASE_NUMERIC = [c for c in model_in.columns if c not in [TARGET, "file_month"] + CAT_COLS]
RAW_COLS = BASE_NUMERIC + CAT_COLS + [TARGET, "CloseDate"]

work = full[RAW_COLS].copy()

# a few rows still can't be used (blank/0 sqft, no lat/long, "missing" as text in the zip), so drop those
good_zip = work.PostalCode.astype(str).str.fullmatch(r"\d{5}")
good_area = work.LivingArea.notna() & (work.LivingArea > 0)
good_loc = work.Latitude.notna() & work.Longitude.notna()
print(f"Dropping: {(~good_area).sum():,} bad LivingArea | {(~good_loc).sum():,} no lat/long | {(~good_zip).sum():,} bad ZIP (rows can overlap)")
work = work[good_zip & good_area & good_loc].copy()
work["PostalCode"] = work.PostalCode.astype(int)
work = work.reset_index(drop=True)

print(full.shape)

Dropping: 363 bad LivingArea | 444 no lat/long | 2 bad ZIP (rows can overlap)
(358166, 71)


In [12]:
# 2. school district
# match each house's lat/long to a school district polygon, then grab the district's
# enrollment + % economically disadvantaged. skips itself if the shapefile isn't there
SD_COLS = []

def add_school_district(d, path):
    import geopandas as gpd
    polys = gpd.read_file(path)
    polys = polys[polys.DistrictTy.isin(["High", "Unified"])].to_crs(4326)
    polys["geometry"] = polys.make_valid()
    pts = gpd.GeoDataFrame(index=d.index, geometry=gpd.points_from_xy(d.Longitude, d.Latitude), crs=4326)
    j = gpd.sjoin(pts, polys[["CDSCode", "EnrollTota", "SEDpct", "geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")].reindex(d.index)
    print(f"School district matched: {j.CDSCode.notna().mean():.2%}")
    return pd.DataFrame({"school_district_geo": j.CDSCode.fillna("missing").astype(str),
                         "sd_enroll": j.EnrollTota.astype(float), "sd_sed_pct": j.SEDpct.astype(float)})

if SCHOOL_DISTRICT_PATH.exists():
    work = work.join(add_school_district(work, SCHOOL_DISTRICT_PATH))
    SD_COLS = ["school_district_geo", "sd_enroll", "sd_sed_pct"]
else:
    print("District shapefile not found - skipping")

School district matched: 99.92%


In [13]:
# 3. split by date
# last month = test, month before = val. everything older is just "history" for now,
# each training window (6, 9, 12... months) picks its own slice of it later
# (if the last month isn't finished yet it gets dropped first)
work["close_month"] = work.CloseDate.dt.to_period("M")
work["mi"] = work.CloseDate.dt.year * 12 + work.CloseDate.dt.month

last_month = work.close_month.max()
if work.CloseDate.max().normalize() < last_month.end_time.normalize():
    last_month = last_month - 1
    work = work[work.close_month <= last_month].copy()
TEST_MONTH, VAL_MONTH = last_month, last_month - 1
work["split"] = np.where(work.close_month == TEST_MONTH, "test", np.where(work.close_month == VAL_MONTH, "val", "history"))
VAL_MI = int(work.loc[work.close_month == VAL_MONTH, "mi"].iloc[0])  # month number of the val month

print(f"History to {VAL_MONTH - 1} | Val {VAL_MONTH} | Test {TEST_MONTH}")
work.groupby("split").size().rename("rows").to_frame()

History to 2026-06 | Val 2026-07 | Test 2026-08


,rows
split,
history,333258
test,11352
val,12748


In [14]:
# 4. area aggregates (prior months only)
# for each house: median $/sqft of sales in its zip + county over the previous 12 months
# (never its own month or later, and test sales never go in the pool). also makes a rough price guess.
# this only looks backward in time so it doesn't depend on the training window, which means we can do it once here
def add_area_aggregates(d, pool_mask, window=AREA_WINDOW, lag=AREA_LAG, short=AREA_SHORT, min_n=AREA_MIN_N):
    d = d.copy()
    d["_ppsf"] = d.ClosePrice / d.LivingArea
    tmp = ["_zip_med", "_zip_n", "_cty_med", "_cty_3m", "_glob"]
    for c in tmp:
        d[c] = np.nan
    pool = d.loc[pool_mask, ["mi", "PostalCode", "CountyOrParish", "_ppsf"]]

    for m in np.sort(d.mi.unique()):
        rows = d.index[d.mi == m]
        win = pool[(pool.mi >= m - window) & (pool.mi <= m - lag)]
        if win.empty:
            continue
        z = win.groupby("PostalCode")["_ppsf"].agg(["median", "count"])
        c12 = win.groupby("CountyOrParish")["_ppsf"].median()
        c3 = win[win.mi >= m - lag - short + 1].groupby("CountyOrParish")["_ppsf"].median()
        zp, cp = d.loc[rows, "PostalCode"], d.loc[rows, "CountyOrParish"]
        d.loc[rows, "_zip_med"] = zp.map(z["median"]).to_numpy()
        d.loc[rows, "_zip_n"] = zp.map(z["count"]).to_numpy()
        d.loc[rows, "_cty_med"] = cp.map(c12).to_numpy()
        d.loc[rows, "_cty_3m"] = cp.map(c3).to_numpy()
        d.loc[rows, "_glob"] = win["_ppsf"].median()

    zip_med = d._zip_med.where(d._zip_n.fillna(0) >= min_n)
    cty_med = d._cty_med.fillna(d._glob)
    est = zip_med.fillna(cty_med)
    # final columns, all from PRIOR months only
    d["zip_n_sales_12m"] = d._zip_n.fillna(0)  # how many zip sales the estimate is based on
    d["zip_med_ppsf_12m"] = zip_med.fillna(est)  # zip median $/sqft (county if zip is too small)
    d["county_med_ppsf_12m"] = cty_med  # county median $/sqft, past year
    d["county_med_ppsf_3m"] = d._cty_3m.fillna(cty_med)  # same but just the last 3 months
    d["county_momentum"] = (d.county_med_ppsf_3m / cty_med).fillna(1.0)  # >1 = prices heating up
    d["area_ppsf_est"] = est  # our best $/sqft guess for the area
    d["est_value_area"] = d.LivingArea * est  # rough price guess = sqft x area $/sqft
    d["log_est_value_area"] = np.log(d.est_value_area)  # logged version
    return d.drop(columns=tmp + ["_ppsf"])

feat = add_area_aggregates(work, pool_mask=(work.split != "test"))
pool_src = feat.loc[feat.split != "test", ["mi", "PostalCode", "ClosePrice", "LivingArea"]].copy()

# the very first month of the data has nothing to look back on, so drop it
n0 = len(feat)
feat = feat[feat.mi >= feat.mi.min() + WARMUP_MONTHS].reset_index(drop=True)
AREA_FEATS = ["zip_n_sales_12m", "zip_med_ppsf_12m", "county_med_ppsf_12m", "county_med_ppsf_3m",
              "county_momentum", "area_ppsf_est", "est_value_area", "log_est_value_area"]
print(f"Warm-up: dropped {n0 - len(feat):,} rows from the first {WARMUP_MONTHS} month(s) of the data")
print(f"Months of history available for training windows: {feat.loc[feat.split == 'history', 'mi'].nunique()}")

Warm-up: dropped 8,303 rows from the first 1 month(s) of the data
Months of history available for training windows: 29


In [16]:
# 5. row-level feature function
# new columns for each house using just its own info: age, size/layout, hoa, time of year,
# map zone, school district size. also fills blanks using the train medians (passed in as "state").
# this only defines the function, it gets run once per training window below
IMPUTE_RAW = ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "LotSizeSquareFeet", "MainLevelBedrooms",
              "ParkingTotal", "Stories", "YearBuilt", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
              "PoolPrivateYN", "ViewYN", "has_hoa", "hoa_monthly"] + [c for c in SD_COLS if c != "school_district_geo"]

def project_km(lat, lon):
    return np.c_[(np.asarray(lon) + 120.0) * 111.32 * np.cos(np.radians(37.0)), np.asarray(lat) * 110.57]

def build_row_features(d, state):
    d = d.copy()
    for c in state["flag_cols"]:
        d[f"{c}_missing"] = d[c].isna().astype("int8")  # 1 = this was blank before we filled it
    for c in IMPUTE_RAW:
        d[c] = d[c].fillna(d["CountyOrParish"].map(state["group_medians"][c])).fillna(state["global_medians"][c])

    # house age stuff
    sale_year = d.CloseDate.dt.year
    d["age_at_sale"] = (sale_year - d.YearBuilt).clip(lower=0)  # how old it was when it sold, no negatives
    d["presale_construction"] = (d.YearBuilt > sale_year).astype("int8")  # 1 = sold before it was finished
    d["age_sq"] = d.age_at_sale ** 2  # so very old and very new can both matter
    d["age_band"] = pd.cut(d.age_at_sale, bins=[-1, 5, 15, 30, 50, 75, np.inf], labels=False).astype("int8")  # buckets 0-5, newer to older

    # size + layout
    d["log_living_area"] = np.log1p(d.LivingArea)  # logs so huge homes dont dominate
    d["log_lot_size"] = np.log1p(d.LotSizeSquareFeet)
    d["living_to_lot_ratio"] = (d.LivingArea / d.LotSizeSquareFeet).clip(upper=5)  # how much of the lot is house
    d["sqft_per_bedroom"] = d.LivingArea / d.BedroomsTotal.clip(lower=1)  # clip so no divide by 0
    d["sqft_per_bath"] = d.LivingArea / d.BathroomsTotalInteger.clip(lower=1)
    d["baths_per_bedroom"] = d.BathroomsTotalInteger / d.BedroomsTotal.clip(lower=1)
    d["total_rooms"] = d.BedroomsTotal + d.BathroomsTotalInteger  # beds + baths only, no living rooms in the data
    d["parking_per_bedroom"] = d.ParkingTotal / d.BedroomsTotal.clip(lower=1)
    d["is_multi_story"] = (d.Stories >= 2).astype("int8")  # 2+ floors
    d["amenity_score"] = d.ViewYN + d.PoolPrivateYN + d.FireplaceYN + d.AttachedGarageYN  # how many of the 4 perks it has
    floor_cols = [c for c in d.columns if c.startswith("floor_") and c != "floor_unknown"]
    d["n_floor_types"] = d[floor_cols].sum(axis=1)  # how many kinds of flooring
    d["has_premium_floor"] = ((d.floor_wood + d.floor_stone) > 0).astype("int8")  # wood or stone somewhere
    d["hoa_log"] = np.log1p(d.hoa_monthly)  # hoa fee, logged
    d["hoa_per_sqft"] = d.hoa_monthly / d.LivingArea  # hoa fee relative to house size

    # time stuff
    d["month_sin"] = np.sin(2 * np.pi * d.CloseDate.dt.month / 12)
    d["month_cos"] = np.cos(2 * np.pi * d.CloseDate.dt.month / 12)  # sin + cos make dec and jan "close" to each other
    d["quarter"] = d.CloseDate.dt.quarter  # 1-4
    d["time_idx"] = d.mi - state["t0"]  # months since training started, picks up market trend

    # location
    d["geo_cluster"] = state["kmeans"].predict(project_km(d.Latitude, d.Longitude)).astype(str)  # which of the 80 map zones it falls in
    d["PostalCode_cat"] = d.PostalCode.astype(str)  # zip as text so we can target encode it later
    if "sd_enroll" in d:
        d["sd_enroll_log"] = np.log1p(d.sd_enroll)  # district size, logged
    return d


In [17]:
# 6. run one training window
# for a window of X months: train = the X months right before val, val = 1 month, test = 1 month (val + test are not counted in X).
# everything that learns from data (blank-filling medians, map zones, target encoding) is fit on that window's train rows only,
# then everything gets saved to fe_outputs/train_{X}m/
FAMILIES_BY_X = {}

def run_window(X):
    d = feat[feat.mi >= VAL_MI - X].copy()   # drop anything older than the window
    d["split"] = np.where(d.close_month == TEST_MONTH, "test", np.where(d.close_month == VAL_MONTH, "val", "train"))
    train0 = d[d.split == "train"]
    if train0.mi.nunique() < X:
        print(f"{X} months: skipped, only {train0.mi.nunique()} months of history available")
        return None

    # fit on train only
    state = {
        "flag_cols": [c for c in IMPUTE_RAW if train0[c].isna().mean() >= MISSING_FLAG_THRESHOLD],
        "group_medians": {c: train0.groupby("CountyOrParish")[c].median() for c in IMPUTE_RAW},
        "global_medians": train0[IMPUTE_RAW].median(),
        "t0": int(train0.mi.min()),
        "kmeans": MiniBatchKMeans(n_clusters=N_GEO_CLUSTERS, random_state=SEED, n_init=3, batch_size=4096)
                  .fit(project_km(train0.Latitude, train0.Longitude)),
    }
    d = build_row_features(d, state)

    # target encoding (avg log price per category, train uses 5-fold cv so a row never sees its own price)
    te_cols = ["City", "PostalCode_cat", "CountyOrParish", "HighSchoolDistrict", "MLSAreaMajor", "geo_cluster"] + \
              [c for c in SD_COLS if c == "school_district_geo"]
    for c in te_cols:
        d[c] = d[c].fillna("missing").astype(str)
    is_tr, is_va, is_te = (d.split == s for s in ["train", "val", "test"])
    encoder = TargetEncoder(target_type="continuous", smooth="auto", cv=5, shuffle=True, random_state=SEED)
    te_feats = [f"{c}_te" for c in te_cols]  # new cols, one "_te" per category column
    d.loc[is_tr, te_feats] = encoder.fit_transform(d.loc[is_tr, te_cols], np.log(d.loc[is_tr, TARGET]))
    d.loc[~is_tr, te_feats] = encoder.transform(d.loc[~is_tr, te_cols])
    d[te_feats] = d[te_feats].astype(float)

    # feature families + checks
    families = {
        "baseline_original": BASE_NUMERIC,
        "missingness": [f"{c}_missing" for c in state["flag_cols"]],
        "intrinsic_eng": ["age_at_sale", "presale_construction", "age_sq", "age_band", "log_living_area", "log_lot_size",
                          "living_to_lot_ratio", "sqft_per_bedroom", "sqft_per_bath", "baths_per_bedroom", "total_rooms",
                          "parking_per_bedroom", "is_multi_story", "amenity_score", "n_floor_types", "has_premium_floor",
                          "hoa_log", "hoa_per_sqft"],
        "temporal_eng": ["month_sin", "month_cos", "quarter", "time_idx"],
        "school_district": [c for c in ["sd_enroll_log", "sd_sed_pct"] if c in d],
        "area_aggregates": AREA_FEATS,
        "target_encoded": te_feats,
    }
    families = {k: v for k, v in families.items() if v}
    all_features = [c for fam in families.values() for c in fam]

    assert len(all_features) == len(set(all_features))
    assert not {"ListPrice", "OriginalListPrice", "DaysOnMarket", "PurchaseContractDate", "BuyerAgencyCompensation"} & set(all_features)
    assert not any("Agent" in c or "Office" in c for c in all_features)
    assert d[all_features].isna().sum().sum() == 0
    assert np.isfinite(d[all_features].to_numpy(dtype=float)).all()
    assert d.loc[is_tr, "mi"].max() < d.loc[is_va, "mi"].min() <= d.loc[is_va, "mi"].max() < d.loc[is_te, "mi"].min()

    # save
    out = OUT_DIR / f"train_{X}m"
    out.mkdir(exist_ok=True)
    for name, part in {"train": d[is_tr], "val": d[is_va], "test": d[is_te]}.items():
        part[["CloseDate", TARGET] + all_features].to_csv(out / f"features_{name}.csv", index=False)

    FAMILIES_BY_X[X] = families
    print(f"{X} months | train {train0.close_month.min()} to {train0.close_month.max()} | {is_tr.sum():,} train, {is_va.sum():,} val, {is_te.sum():,} test | {len(all_features)} features")
    return {"train_months": X, "train_rows": int(is_tr.sum()), "val_rows": int(is_va.sum()),
            "test_rows": int(is_te.sum()), "n_features": len(all_features)}

In [18]:
# 7. run every training window
# runs the function above for 6, 9, 12... months and shows one summary row per window
results = []
for X in TRAIN_WINDOWS:
    r = run_window(X)
    if r:
        results.append(r)
pd.DataFrame(results)

6 months | train 2026-01 to 2026-06 | 64,566 train, 12,748 val, 11,352 test | 81 features
9 months | train 2025-10 to 2026-06 | 96,739 train, 12,748 val, 11,352 test | 81 features
12 months | train 2025-07 to 2026-06 | 131,702 train, 12,748 val, 11,352 test | 81 features
15 months | train 2025-04 to 2026-06 | 166,979 train, 12,748 val, 11,352 test | 81 features
18 months | train 2025-01 to 2026-06 | 194,532 train, 12,748 val, 11,352 test | 81 features
21 months | train 2024-10 to 2026-06 | 228,140 train, 12,748 val, 11,352 test | 81 features
24 months | train 2024-07 to 2026-06 | 264,762 train, 12,748 val, 11,352 test | 81 features
27 months | train 2024-04 to 2026-06 | 303,805 train, 12,748 val, 11,352 test | 81 features
30 months: skipped, only 29 months of history available


,train_months,train_rows,val_rows,test_rows,n_features
0,6,64566,12748,11352,81
1,9,96739,12748,11352,81
2,12,131702,12748,11352,81
3,15,166979,12748,11352,81
4,18,194532,12748,11352,81
5,21,228140,12748,11352,81
6,24,264762,12748,11352,81
7,27,303805,12748,11352,81


In [19]:
# 8. included vs excluded columns
# shows which columns made it into the files (and their family) and which got left out
# (uses the largest window that ran, the columns are the same except the missing-flags can differ a little between windows)
FAMILIES = FAMILIES_BY_X[results[-1]["train_months"]]
ALL_FEATURES = [c for fam in FAMILIES.values() for c in fam]
out_cols = ["CloseDate", TARGET] + ALL_FEATURES

included = pd.DataFrame(
    [("CloseDate", "id"), (TARGET, "target")] + [(c, fam) for fam, cols in FAMILIES.items() for c in cols],
    columns=["column", "family"],
)

excluded = pd.DataFrame({"column": [c for c in full.columns if c not in out_cols]})
excluded["reason"] = excluded.column.map(
    lambda c: "raw text, replaced by target-encoded version" if c in CAT_COLS else "not used"
)

with pd.option_context("display.max_rows", None):
    print(f"INCLUDED ({len(included)})")
    display(included)
    print(f"EXCLUDED from cleaned_all.csv ({len(excluded)})")
    display(excluded)

INCLUDED (83)


,column,family
0,CloseDate,id
1,ClosePrice,target
2,hoa_monthly,baseline_original
3,BathroomsTotalInteger,baseline_original
4,BedroomsTotal,baseline_original
5,GarageSpaces,baseline_original
6,Latitude,baseline_original
7,LivingArea,baseline_original
8,Longitude,baseline_original
9,LotSizeSquareFeet,baseline_original


EXCLUDED from cleaned_all.csv (37)


,column,reason
0,OriginalListPrice,not used
1,ListingKey,not used
2,ListAgentEmail,not used
3,ListAgentFirstName,not used
4,ListAgentLastName,not used
5,UnparsedAddress,not used
6,PropertyType,not used
7,ListPrice,not used
8,DaysOnMarket,not used
9,ListOfficeName,not used
